# Tutorial 15: Camera Defects and Cosmic-Ray Artifacts

This tutorial adds sensor-like artifacts to arbitrary 2-D images while keeping camera defects consistent across an image collection.

The sample-to-detector selector below defaults to the fast Fraunhofer FFT. See [Tutorial 16](16_compare_detector_propagation.ipynb) for the same-exit-wave Rayleigh–Sommerfeld comparison. This notebook stops before detector propagation or operates on supplied images; the selector is provided for extending the workflow. 


In [ ]:
# Sample-to-detector propagation (independent of multislice).
detector_propagation_method = "fraunhofer"  # Default; opt in with "rayleigh_sommerfeld".
# Direct Rayleigh-Sommerfeld is expensive: try small grids first.
# This tutorial does not propagate to a detector; pass this setting to
# DetectorConfig/HologramPipelineConfig when extending it to generate holograms.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
from scattering_calculator.experimental_conditions.detector import add_sensor_artifacts


## 1. Two random processes with different lifetimes

**Camera-persistent defects:** hot and cold counts are sampled once from Poisson distributions. Their averages are user settings. A stable `camera_seed` reproduces the same coordinates and each pixel's initial baseline value. Baseline values vary across pixels by `*_value_spread`; repeated exposures vary around each assigned baseline by `*_temporal_sigma`. Both controls are fractional standard deviations.

**Exposure-transient events:** cosmic-ray count is `N ~ Poisson(cosmic_rays_per_second × effective_exposure_seconds)`. Every exposure can use a different `exposure_seed`. Each event is a smoothly blurred Gaussian ellipse whose major-axis FWHM, aspect ratio, direction, position, and peak value are sampled independently. When exposure is `None`, the effective exposure is 1 second. A detector's maximum-count setting does not replace this rate.


## 2. Configure one simulated camera

`flicker_fraction` selects a stable subset of hot-pixel coordinates as unstable. `flicker_probability` controls how often they activate; their positions and baseline values never move. `cosmic_ray_length_range` controls the short major-axis FWHM in pixels, while `cosmic_ray_aspect_ratio_range=(2, 3)` makes every event independently vary between those ratios.


In [ ]:
camera = {
    "camera_seed": 31415,
    "average_hot_pixels": 250,
    "average_cold_pixels": 120,
    "flicker_fraction": 0.25,
    "flicker_probability": 0.5,
    "hot_pixel_value": 5000,
    "hot_pixel_value_spread": 0.08,
    "hot_pixel_temporal_sigma": 0.03,
    "cold_pixel_value": 50,
    "cold_pixel_value_spread": 0.20,
    "cold_pixel_temporal_sigma": 0.08,
    "cosmic_rays_per_second": 30.0,
    "cosmic_ray_value": 4500,
    "cosmic_ray_value_spread": 0.15,
    "cosmic_ray_length_range": (2.0, 5.0),
    "cosmic_ray_aspect_ratio_range": (2.0, 3.0),
}


## 3. Apply the same camera to several images

Only `exposure_seed` changes. Bad-pixel locations and baselines repeat, their measured values oscillate around those baselines, and cosmic-ray shapes and peaks differ.


In [ ]:
%matplotlib widget


shape = (256, 256)
y, x = np.indices(shape)
clean = 900 * np.exp(-((x - 128)**2 + (y - 128)**2) / (2 * 48**2))
images, metadata = [], []
for exposure_seed in (10, 11, 12):
    image, info = add_sensor_artifacts(
        clean, artifacts_config=camera, exposure_time=2.0,
        detector_threshold=5000, exposure_seed=exposure_seed,
    )
    images.append(image); metadata.append(info)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.3))
axes[0].imshow(clean, cmap="magma", vmin=0, vmax=5000); axes[0].set_title("clean")
for ax, image, info in zip(axes[1:], images, metadata):
    ax.imshow(image, cmap="magma", vmin=0, vmax=5000)
    ax.set_title(f"{info['cosmic_rays']} cosmic rays")
for ax in axes: ax.set_axis_off()
metadata


## 4. Missing exposure time

Passing `exposure_time=None` uses 1 second, including when an image is normalized using a maximum-count setting and has no timing metadata.


In [ ]:
fallback_image, fallback_info = add_sensor_artifacts(
    clean, artifacts_config=camera, exposure_time=None,
    detector_threshold=5000, exposure_seed=20,
)
fallback_info


## Practical rules

- Keep `camera_seed` and image geometry unchanged for one camera dataset.
- Use a distinct `exposure_seed` per image for independent transient events.
- Poisson means are averages, not exact counts.
- Apply artifacts before resizing or cropping when coordinates represent detector pixels.
- Set a component's average or rate to zero to disable it.
